# Project 03 (final): EDA — who cycles when, and why?

**Scenario:** you work for the bike sharing system **Capital Bikeshare**
(Washington D.C.). Management wants to know: *what does demand depend on? When do we
need the most bikes? Do our customer groups differ?*
Your task: a complete exploratory data analysis (EDA) on the **real** rental data
2011-2012 (17,379 hourly records) — with a conclusion in clear sentences.

**Preparation** (once, in the folder `03-final`, venv active):

```
python datasets/download_data.py
```

**How to work:** this is a final project — the guiding questions are given, but you
write the code entirely yourself (hints are given as prose; the reference solution is
in `solution/`). Stick to the EDA workflow from script 2.5: overview → quality →
univariate → bivariate → time → conclusion. **Every figure gets a sentence of
interpretation.**

## 1. Overview and documentation

Columns (from the official data set description):

| Column | Meaning |
|--|--|
| `dteday`, `hr` | date, hour (0-23) |
| `season` | 1 winter, 2 spring, 3 summer, 4 autumn |
| `yr` | 0 = 2011, 1 = 2012 |
| `holiday`, `weekday`, `workingday` | holiday, day of the week (0 = Sunday), working day |
| `weathersit` | 1 clear, 2 mist/clouds, 3 light rain/snow, 4 severe weather |
| `temp`, `atemp` | temperature / apparent — **normalised!** `temp*41` = degrees C, `atemp*50` = degrees C |
| `hum`, `windspeed` | humidity (`*100` = %), wind (`*67` = km/h) — normalised |
| `casual`, `registered`, `cnt` | casual users, registered users, total |

**Task:** load `datasets/hour.csv`, do the first inspection (`shape`, `head`, `info`)
and create **de-normalised** columns: `temp_c`, `hum_pct`, `wind_kmh`.
*(Lesson: without the documentation you would have analysed "temperatures" between 0 and 1.)*

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


## 2. Data quality

**Tasks:**
1. Missing values? Duplicates? (Spoiler: the data set is well kept — but check!)
2. Consistency: does `cnt == casual + registered` hold everywhere?
3. **The hidden gap:** 2011-2012 have 17,544 hours, the data set has 17,379 rows.
   Are hours missing — and if so, how many? (Hint: build a complete hourly index with
   `pd.date_range(..., freq="h")` and compare.)

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Finding:** no NaN, no duplicates, `cnt` consistent — but **165 hours are missing
entirely** (operational outages/maintenance). For our mean-based analyses this is
uncritical; for a gap-free time series analysis (a later module) the gaps would have to
be handled explicitly. **Note: "no NaN" does not mean "nothing missing" — missing ROWS
are only seen by those who look for them.**

## 3. Univariate: understanding the target quantity

**Tasks:**
1. Histogram of `cnt` (rentals per hour) — describe the shape.
2. Key figures: mean vs. median — does that fit the shape? (Script 1.3!)

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Interpretation:** strongly **right-skewed** — many quiet hours (night!), few peak
hours of almost 1000 rentals. Mean (189) > median (142), just as the script predicts for
right-skewed distributions. For a "typical hour" the median is the more honest number.

## 4. Time patterns — the core question of the operation

**Tasks:**
1. Mean rentals per hour (`groupby("hr")`) — as a line plot.
2. The same split by `workingday` (0/1) — two lines in one plot.
   (Hint: `df.groupby(["hr", "workingday"])["cnt"].mean().unstack()`)
3. Describe the patterns: when are the peaks? Why do they differ?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Interpretation — the most important plot of the project:** on **working days** two
sharp peaks around **8 a.m.** (about 480) and **5-6 p.m.** (about 525/460): commuter
traffic. On **days off** a single flat hump in the afternoon: leisure traffic.
These are two completely different modes of use in one system — capacity and
rebalancing have to follow the working-day profile.

## 5. Two customer groups: casual vs. registered

**Task:** repeat the hourly profile, but compare `casual` and `registered`
(working days only). Which group produces the commuter peaks? What does that mean for
marketing (who do you reach when)?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Interpretation:** the commuter peaks come almost entirely from **registered** users
(subscriptions = everyday traffic). **Casual** users ride spread flatly over the
afternoon — tourists/leisure. Two customer groups, two ways of addressing them.

## 6. Weather

**Tasks:**
1. Scatter plot `temp_c` vs. `cnt` (choose a small alpha, 17k points!) and the
   Pearson correlation.
2. Box plot of `cnt` by `weathersit`. Careful: how often does category 4 occur?
   (`value_counts()`) — what does that mean for the informative value of that box?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Interpretation:** temperature correlates positively (r about 0.40) — but the scatter
shows: at EVERY temperature there are also hours of weak demand (namely at night).
The hour of the day overlays the weather effect. Worse weather visibly lowers demand;
**category 4 (severe weather), however, occurs only 3 times** — that box plot is
statistically worthless (n = 3!). Always check group sizes before comparing box plots.

## 7. The aggregation lesson

**Task:** compute r(temperature, rentals) once more — but at the **daily level**
(`day.csv` is already in the datasets folder, or aggregate yourself with
`groupby("dteday")`). Compare with the hourly value from above. Why is the daily value
so much higher?

In [ ]:
# Your code here. (Reference solution: solution/solution.ipynb)


**Interpretation:** r jumps from **0.41 to 0.63**. At the hourly level the day-night
cycle adds noise to the relationship (3 a.m. hours are always empty, no matter how warm).
At the daily level this averages out. **Lesson: correlations depend on the level of
aggregation** — there is no such thing as "the" correlation between two quantities, you
have to say what counts as one observation (related to Simpson, script 3.2).

## 8. Your conclusion for management

**Task:** write 5 bullet points here in everyday language (no technical terms!) that
summarise the most important findings — the way you would send them to management in
an email.

*Your answer:*

- ...

<details><summary>Reference conclusion (formulate your own first!)</summary>

- Demand is dominated by commuters: on weekdays two sharp peaks around 8 a.m. and 5-6 p.m. — that is when bikes have to be at stations and offices.
- At the weekend everything shifts to the afternoon and into the parks and leisure areas — redistributing the bikes has to work differently on weekdays than at the weekend.
- Registered users = commuters, casual users = leisure/tourists. Advertise subscriptions in the morning at transport hubs, day tickets at midday at the sights.
- Warmth and good weather drive demand noticeably (on a daily basis one of the strongest factors) — the weather forecast belongs in capacity planning.
- The business is growing strongly: 2012 was about 65 % above 2011. Plan capacity for the next season accordingly.
</details>

## Done — what you can do now

- carry out a complete EDA following the workflow: overview → quality → univariate →
  bivariate → time → conclusion
- read documentation and recognise normalised variables (temp*41!)
- find hidden gaps in data (missing rows instead of NaN)
- discover patterns in time series and explain them **substantively** (commuters!)
- assess correlations critically (level of aggregation, group sizes)
- summarise results appropriately for the audience

**Bonus tasks:**
1. Holidays: does a public holiday on a Monday behave like a Sunday? (`holiday`)
2. Humidity and wind: is either of these variables worth using as a demand indicator?
3. Build a correlation matrix (`df[[...]].corr()`, `sns.heatmap`) — and name one
   correlation in it that must NOT be interpreted causally.